# AIRPATH-AI — Milestone 2A forecasting baselines

This notebook constructs leakage-safe station-level samples for t+1h, t+2h, and t+3h and evaluates persistence and training-only historical-time baselines. It does not interpolate, fit a learned model, or use spatial/routing information.

In [1]:
from pathlib import Path
import sys

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

from src.baselines import evaluate_baselines, write_baseline_outputs
from src.data_loading import load_air_quality_csv
from src.data_validation import audit_dataset
from src.forecasting_data import build_forecasting_samples, split_boundaries

DATA_PATH = ROOT / "data" / "raw" / "Air Quality Ho Chi Minh City.csv"

In [2]:
raw = load_air_quality_csv(DATA_PATH)
clean, _ = audit_dataset(raw)
boundaries = split_boundaries(clean)
samples, sample_counts = build_forecasting_samples(clean)
predictions, metrics = evaluate_baselines(samples, clean)

display(boundaries)
display(sample_counts)
print(f"Valid station-horizon samples: {len(samples):,}")

,split,start,end,unique_timestamps
0,train,2021-02-23 21:00:00,2022-01-31 02:00:00,7316
1,validation,2022-01-31 03:00:00,2022-04-08 00:00:00,1568
2,test,2022-04-08 01:00:00,2022-06-21 17:00:00,1568


,Station_No,horizon_hours,origin_observations,complete_lag_origins,exact_nonmissing_targets,cross_partition_candidates,valid_samples
0,1,1,7892,7723,7834,2,7666
1,2,1,9357,9161,9289,1,9099
2,3,1,8418,8123,8316,1,8028
3,4,1,9951,9655,9847,1,9560
4,5,1,7431,7266,7372,2,7213
5,6,1,9499,9270,9416,1,9201
6,1,2,7892,7723,7791,4,7623
7,2,2,9357,9161,9230,2,9042
8,3,2,8418,8123,8251,2,7964
9,4,2,9951,9655,9791,2,9505


Valid station-horizon samples: 151,400


In [3]:
overall_and_horizon = metrics.loc[
    metrics["Station_No"].eq("ALL")
]
station_horizon_test = metrics.loc[
    metrics["split"].eq("test")
    & ~metrics["Station_No"].eq("ALL")
    & ~metrics["horizon_hours"].eq("ALL")
]
display(overall_and_horizon)
display(station_horizon_test)

,model,split,Station_No,horizon_hours,n,mae,rmse,r2
0,persistence,validation,ALL,ALL,25744,4.486610,9.218151,0.399007
1,persistence,validation,ALL,1,8603,3.038068,7.025280,0.651794
2,persistence,validation,ALL,2,8580,4.639194,9.453301,0.367451
3,persistence,validation,ALL,3,8561,5.789336,10.787420,0.175577
28,persistence,test,ALL,ALL,23655,4.316185,8.746862,0.417159
29,persistence,test,ALL,1,7929,2.999557,7.140499,0.613373
30,persistence,test,ALL,2,7883,4.494094,8.876511,0.399070
31,persistence,test,ALL,3,7843,5.468435,10.000589,0.235379
56,historical_time,validation,ALL,ALL,25744,8.266889,11.247184,0.105317
57,historical_time,validation,ALL,1,8603,8.271190,11.258258,0.105766


,model,split,Station_No,horizon_hours,n,mae,rmse,r2
38,persistence,test,1,1,1540,3.151632,5.376727,0.805526
39,persistence,test,1,2,1533,4.816937,7.998415,0.570373
40,persistence,test,1,3,1527,6.041208,9.599223,0.380975
41,persistence,test,2,1,1407,2.508453,3.977243,0.715614
42,persistence,test,2,2,1402,3.793885,5.903175,0.370723
43,persistence,test,2,3,1397,4.604210,7.136402,0.074291
44,persistence,test,3,1,974,3.568307,5.556596,0.582791
45,persistence,test,3,2,965,5.164001,7.745137,0.172775
46,persistence,test,3,3,957,6.143315,9.007654,-0.125577
47,persistence,test,4,1,1404,4.190050,6.779909,0.725480


In [4]:
write_baseline_outputs(
    ROOT, samples, sample_counts, boundaries, predictions, metrics
)
print("Forecasting samples, predictions, metrics, and report written.")

Forecasting samples, predictions, metrics, and report written.


## Interpretation safeguards

- Targets and lags use exact timestamp matching; rows across missing timestamps are never shifted into false hourly neighbors.
- Samples whose target crosses a split boundary are excluded.
- Historical means are fitted from training observations only.
- Zero and IQR-flagged PM2.5 observations remain present for later sensitivity analysis.
- Validation supports development; test results should remain held out from repeated model selection.